# 🥗 Food Calorie Predictor — TF-IDF Model Training Notebook
**Model:** TF-IDF + Cosine Similarity  
**Dataset:** 13,829 food items with name, ingredients, category, and nutritional info  
**Task:** Given a food name → return calories + health warning

## 1. Install & Import Libraries

In [31]:
# Install required packages
!pip install scikit-learn pandas numpy joblib

In [32]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('All libraries loaded successfully!')

All libraries loaded successfully!


## 2. Load & Explore the Dataset

In [33]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/nourannhassann/dataset-food/final_matched_dataset (1).csv


In [34]:
import os

print(os.listdir("/kaggle/input/datasets/nourannhassann"))

['dataset-food']


In [35]:
print(os.listdir("/kaggle/input/datasets/nourannhassann/dataset-food"))

['final_matched_dataset (1).csv']


In [36]:
import pandas as pd

file_path = "/kaggle/input/datasets/nourannhassann/dataset-food/final_matched_dataset (1).csv"

df = pd.read_csv(file_path)

print(df.shape)
df.head()

(13829, 9)


,name,matched_name,ingredients,category,portion,calories,protein,fat,carb
0,chicken handi,chicken,"Chicken, Onion, Tomatoes, Garlic, Ginger paste...",meat,100 g,166.0,21.4,1.79,0.0
1,chicken mandi,chicken,"Chicken, Basmati Rice, Water, Onion, Garlic, G...",meat,100 g,166.0,21.4,1.79,0.0
2,sticky chicken,chicken,"Chicken drumsticks, Soy Sauce, Honey, Olive Oi...",meat,100 g,166.0,21.4,1.79,0.0
3,chicken congee,chicken,"Chicken, Salt, Pepper, Ginger Cordial, Ginger,...",meat,100 g,166.0,21.4,1.79,0.0
4,chicken karaage,chicken,"Chicken, Ginger, Garlic, Soy sauce, Sake, Gran...",meat,100 g,166.0,21.4,1.79,0.0


In [37]:
# Calorie statistics
print('=== Calorie Stats ===')
print(df['calories'].describe())

print('\n=== Calorie Distribution ===')
bins = pd.cut(df['calories'], bins=[0,100,200,350,500,700,929])
print(bins.value_counts().sort_index())

=== Calorie Stats ===
count    13829.000000
mean       206.839395
std        164.020339
min          0.000000
25%         76.000000
50%        173.000000
75%        308.000000
max        929.000000
Name: calories, dtype: float64

=== Calorie Distribution ===
calories
(0, 100]      4339
(100, 200]    3573
(200, 350]    3235
(350, 500]    1875
(500, 700]     521
(700, 929]     195
Name: count, dtype: int64


In [38]:
# Top categories
print('=== Top 10 Categories ===')
print(df['category'].value_counts().head(10))

print(f'\nTotal unique food names: {df["name"].nunique():,}')
print(f'Null values:\n{df.isnull().sum()}')

=== Top 10 Categories ===
category
vegetables-legumes                1554
meals-dishes                      1488
sauces-gravy-dressing-spreads      984
cakes-pies                         948
fruit                              828
meat                               787
fish-seafood                       679
salad                              646
sweets-chocolate-cookies-candy     588
bread-rolls-pastries               473
Name: count, dtype: int64

Total unique food names: 13,553
Null values:
name            5
matched_name    0
ingredients     0
category        0
portion         0
calories        0
protein         0
fat             0
carb            0
dtype: int64


## 3. Data Preprocessing

In [39]:
# Clean the data
df = df.dropna(subset=['calories'])

for col in ['name', 'matched_name', 'ingredients', 'category']:
    df[col] = df[col].fillna('')

print(f'Rows after cleaning: {len(df):,}')

# Build rich text feature
# name is repeated 3x for higher weight (most important signal)
# matched_name repeated 2x
# then category and ingredients
def build_text(row):
    n   = row['name'].lower().strip()
    mn  = row['matched_name'].lower().strip()
    ing = row['ingredients'].lower()
    cat = row['category'].lower().replace('-', ' ')
    return f"{n} {n} {n} {mn} {mn} {cat} {ing}"

df['text'] = df.apply(build_text, axis=1)

print('\nSample text feature (row 0):')
print(df['text'].iloc[0][:300])

Rows after cleaning: 13,829

Sample text feature (row 0):
chicken handi chicken handi chicken handi chicken chicken meat chicken, onion, tomatoes, garlic, ginger paste, vegetable oil, cumin seeds, coriander seeds, turmeric powder, chilli powder, green chilli, yogurt, cream, fenugreek, garam masala, salt


## 4. Train the TF-IDF Model

In [40]:
# Train TF-IDF Vectorizer
# ngram_range=(1,3) captures unigrams, bigrams, trigrams
# sublinear_tf=True applies log scaling to term frequencies
# min_df=1 keeps rare terms (every food name matters)
# max_df=0.95 removes terms appearing in >95% of docs (too common)

print('Training TF-IDF vectorizer...')
vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

tfidf_matrix = vectorizer.fit_transform(df['text'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  Rows    = {tfidf_matrix.shape[0]:,} food items')
print(f'  Columns = {tfidf_matrix.shape[1]:,} unique n-gram features')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

Training TF-IDF vectorizer...
TF-IDF matrix shape: (13829, 446119)
  Rows    = 13,829 food items
  Columns = 446,119 unique n-gram features
Vocabulary size: 446,119


## 5. Save the Model

In [41]:
# Save all model artifacts
os.makedirs('model', exist_ok=True)

joblib.dump(vectorizer,    'model/tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix,  'model/tfidf_matrix.pkl')
df.to_csv('model/food_db.csv', index=False)

print('Model saved!')
print(f'  model/tfidf_vectorizer.pkl  — the trained TF-IDF vectorizer')
print(f'  model/tfidf_matrix.pkl      — the pre-computed document matrix')
print(f'  model/food_db.csv           — the cleaned food database')

Model saved!
  model/tfidf_vectorizer.pkl  — the trained TF-IDF vectorizer
  model/tfidf_matrix.pkl      — the pre-computed document matrix
  model/food_db.csv           — the cleaned food database


## 6. Prediction Function + Calorie Warning System

In [42]:
# Calorie warning thresholds
THRESHOLDS = [
    (0,   100,  'very_low',  '  Very low calorie — great for a light snack!'),
    (100, 200,  'low',       '  Low calorie — a healthy, balanced choice.'),
    (200, 350,  'moderate',  '   Moderate calorie — fine as part of a balanced meal.'),
    (350, 500,  'high',      '   High calorie — enjoy in moderation.'),
    (500, 700,  'very_high', '  Very high calorie — consider a smaller portion!'),
    (700, 9999, 'extreme',   '  ENOUGH! Extremely calorie-dense — eat very sparingly!'),
]

def get_warning(cal):
    for lo, hi, tier, msg in THRESHOLDS:
        if lo <= cal < hi:
            return tier, msg
    return 'extreme', THRESHOLDS[-1][3]

def predict(query, top_k=5):
    q     = query.lower().strip()
    q_vec = vectorizer.transform([q])
    sims  = cosine_similarity(q_vec, tfidf_matrix).flatten()

    # Boost: direct word overlap with food name
    q_words = set(q.split())
    for i, name in enumerate(df['name'].str.lower()):
        overlap = len(q_words & set(str(name).split())) / max(len(q_words), 1)
        sims[i] += overlap * 0.3

    top_idx = sims.argsort()[::-1][:top_k]
    results = []
    for i in top_idx:
        row = df.iloc[i]
        cal = round(float(row['calories']))
        tier, warning = get_warning(cal)
        results.append({
            'food':     row['name'],
            'calories': cal,
            'protein':  round(float(row['protein']), 1),
            'fat':      round(float(row['fat']),     1),
            'carbs':    round(float(row['carb']),    1),
            'category': row['category'],
            'score':    round(float(sims[i]), 3),
            'warning':  warning,
            'tier':     tier,
        })
    return results

print('predict() function ready!')

predict() function ready!


## 7. Test the Model

In [43]:
# Test with sample queries
test_queries = [
    'chicken', 'pizza', 'chocolate cake', 'grilled salmon',
    'banana', 'rice', 'burger', 'sushi', 'olive oil', 'lentils'
]

print(f'{"Query":<20} {"Best Match":<38} {"Calories":<12} Warning')
print('-' * 95)

for q in test_queries:
    r = predict(q, top_k=1)[0]
    print(f'{q:<20} {r["food"]:<38} {str(r["calories"]) + " kcal":<12} {r["warning"]}')

Query                Best Match                             Calories     Warning
-----------------------------------------------------------------------------------------------
chicken              spanish chicken                        166 kcal       Low calorie — a healthy, balanced choice.
pizza                pizza dough                            307 kcal        Moderate calorie — fine as part of a balanced meal.
chocolate cake       chocolate cake                         345 kcal        Moderate calorie — fine as part of a balanced meal.
grilled salmon       grilled salmon with lime butter sauce  185 kcal       Low calorie — a healthy, balanced choice.
banana               banana pancakes                        270 kcal        Moderate calorie — fine as part of a balanced meal.
rice                 coconut rice                           185 kcal       Low calorie — a healthy, balanced choice.
burger               the ultimate burger                    212 kcal        Moderate cal

In [44]:
# Detailed view for a single query
query = 'chicken'   # <-- change this to any food you want

results = predict(query, top_k=5)
best = results[0]

print(f'Query: "{query}"')
print('=' * 55)
print(f'  Best match  : {best["food"].title()}')
print(f'  Calories    : {best["calories"]} kcal / 100g')
print(f'  Protein     : {best["protein"]} g')
print(f'  Fat         : {best["fat"]} g')
print(f'  Carbs       : {best["carbs"]} g')
print(f'  Category    : {best["category"].replace("-", " ").title()}')
print(f'  Match score : {best["score"]}')
print(f'  {best["warning"]}')
print('=' * 55)
print('Other close matches:')
for r in results[1:]:
    print(f'  • {r["food"]:<40} {r["calories"]} kcal  (score={r["score"]})')

Query: "chicken"
  Best match  : Spanish Chicken
  Calories    : 166 kcal / 100g
  Protein     : 21.4 g
  Fat         : 1.8 g
  Carbs       : 0.0 g
  Category    : Meat
  Match score : 0.443
    Low calorie — a healthy, balanced choice.
Other close matches:
  • roast chicken                            166 kcal  (score=0.435)
  • sticky chicken                           166 kcal  (score=0.431)
  • chicken marengo                          166 kcal  (score=0.425)
  • chicken congee                           166 kcal  (score=0.422)


## 8. Load Saved Model (for future use)

In [45]:
# Load the saved model without retraining
vectorizer_loaded   = joblib.load('model/tfidf_vectorizer.pkl')
tfidf_matrix_loaded = joblib.load('model/tfidf_matrix.pkl')
df_loaded           = pd.read_csv('model/food_db.csv')

print(f'Model loaded! {len(df_loaded):,} foods in database.')

# Quick test
q_vec = vectorizer_loaded.transform(['pizza'])
sims  = cosine_similarity(q_vec, tfidf_matrix_loaded).flatten()
best_idx = sims.argmax()
print(f'Test — pizza → {df_loaded.iloc[best_idx]["name"]} ({df_loaded.iloc[best_idx]["calories"]} kcal)')

Model loaded! 13,829 foods in database.
Test — pizza → pizza dough (307.0 kcal)


## 9. Interactive CLI (run in terminal)

In [ ]:
# Run this cell for an interactive food lookup session
print('Food Calorie Predictor — Interactive Mode')
print('Type a food name and press Enter. Type quit to stop.\n')

while True:
    query = input('Enter food > ').strip()
    if query.lower() in ('quit', 'exit', 'q', ''):
        print('Goodbye!')
        break
    results = predict(query, top_k=5)
    if not results:
        print('  No match found.\n')
        continue
    b = results[0]
    print(f'\n  Best match  : {b["food"].title()}')
    print(f'  Calories    : {b["calories"]} kcal / 100g')
    print(f'  Protein     : {b["protein"]}g  |  Fat: {b["fat"]}g  |  Carbs: {b["carbs"]}g')
    print(f'  {b["warning"]}\n')

Food Calorie Predictor — Interactive Mode
Type a food name and press Enter. Type quit to stop.

